<a href="https://colab.research.google.com/github/WINNER03/malarialens/blob/main/MalariaLens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MalariaLens: malaria cell classifier
Fine-tunes MobileNetV2 on the NIH malaria dataset (27,558 cell images) to detect parasitized red blood cells.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all (~15 min).

In [ ]:
!pip -q install gradio importlib_resources
import tensorflow as tf, tensorflow_datasets as tfds, numpy as np
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

TF 2.21.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1. Load the NIH dataset (80% train / 10% val / 10% test)

In [ ]:
(train, val, test), info = tfds.load('malaria', split=['train[:80%]','train[80%:90%]','train[90%:]'],
                                     as_supervised=True, with_info=True)
CLASSES = info.features['label'].names
print(CLASSES, info.splits['train'].num_examples, 'images')

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/malaria/incomplete.MW6H6Z_1.0.0/malaria-train.tfrecord-[0-9][0-9][0-9][0-9…

Dataset malaria downloaded and prepared to /root/tensorflow_datasets/malaria/1.0.0. Subsequent calls will reuse this data.
['parasitized', 'uninfected'] 27558 images


## 2. Preprocess: resize to 128x128

In [ ]:
IMG, BATCH = 128, 64
def prep(x, y):
    return tf.cast(tf.image.resize(x, (IMG, IMG)), tf.float32), tf.cast(y, tf.int32)
train_ds = train.map(prep).shuffle(2000).batch(BATCH).prefetch(2)
val_ds = val.map(prep).batch(BATCH).prefetch(2)
test_ds = test.map(prep).batch(BATCH).prefetch(2)

## 3. Transfer learning with MobileNetV2 + built-in augmentation

In [ ]:
base = tf.keras.applications.MobileNetV2(input_shape=(IMG,IMG,3), include_top=False, weights='imagenet')
base.trainable = False
inputs = tf.keras.Input((IMG,IMG,3))
x = tf.keras.layers.RandomFlip('horizontal_and_vertical')(inputs)
x = tf.keras.layers.RandomRotation(0.2)(x)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(CLASSES), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=3)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/3
345/345 ━━━━━━━━━━━━━━━━━━━━ 45s 82ms/step - accuracy: 0.8337 - loss: 0.3851 - val_accuracy: 0.9057 - val_loss: 0.2471
Epoch 2/3
345/345 ━━━━━━━━━━━━━━━━━━━━ 19s 50ms/step - accuracy: 0.8915 - loss: 0.2690 - val_accuracy: 0.8784 - val_loss: 0.2843
Epoch 3/3
345/345 ━━━━━━━━━━━━━━━━━━━━ 19s 53ms/step - accuracy: 0.8965 - loss: 0.2641 - val_accuracy: 0.9064 - val_loss: 0.2504


## 4. Fine-tune the top layers

In [ ]:
base.trainable = True
for layer in base.layers[:100]: layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=5,
          callbacks=[tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)])

Epoch 1/5
345/345 ━━━━━━━━━━━━━━━━━━━━ 45s 93ms/step - accuracy: 0.8983 - loss: 0.3052 - val_accuracy: 0.8959 - val_loss: 0.2832
Epoch 2/5
345/345 ━━━━━━━━━━━━━━━━━━━━ 31s 85ms/step - accuracy: 0.9224 - loss: 0.2376 - val_accuracy: 0.9086 - val_loss: 0.2580
Epoch 3/5
345/345 ━━━━━━━━━━━━━━━━━━━━ 31s 87ms/step - accuracy: 0.9332 - loss: 0.2043 - val_accuracy: 0.9234 - val_loss: 0.2317
Epoch 4/5
345/345 ━━━━━━━━━━━━━━━━━━━━ 31s 85ms/step - accuracy: 0.9378 - loss: 0.1918 - val_accuracy: 0.9369 - val_loss: 0.1986
Epoch 5/5
345/345 ━━━━━━━━━━━━━━━━━━━━ 30s 85ms/step - accuracy: 0.9413 - loss: 0.1763 - val_accuracy: 0.9365 - val_loss: 0.2050


## 5. Evaluate on held-out test set
Sensitivity matters most: a missed infection is worse than a false alarm.

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = model.predict(test_ds).argmax(1)
P = CLASSES.index('parasitized')
tp = np.sum((y_pred==P)&(y_true==P)); fn = np.sum((y_pred!=P)&(y_true==P))
tn = np.sum((y_pred!=P)&(y_true!=P)); fp = np.sum((y_pred==P)&(y_true!=P))
print(f'Accuracy:    {(tp+tn)/len(y_true):.2%}')
print(f'Sensitivity: {tp/(tp+fn):.2%}  (infected cells caught)')
print(f'Specificity: {tn/(tn+fp):.2%}  (healthy cells correctly cleared)')
print(f'Test images: {len(y_true)}')

44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 68ms/step
Accuracy:    93.76%
Sensitivity: 89.06%  (infected cells caught)
Specificity: 98.41%  (healthy cells correctly cleared)
Test images: 2756


## 6. Save the model

In [ ]:
model.save('malarialens.keras')
from google.colab import files
files.download('malarialens.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Live demo: upload a cell image
Running this gives a public link (valid 72h). Take a screenshot for your GitHub README.

In [ ]:
import gradio as gr
def predict(img):
    x = tf.image.resize(img, (IMG, IMG))[None]
    p = model.predict(x, verbose=0)[0]
    return {CLASSES[i]: float(p[i]) for i in range(len(CLASSES))}
examples = [tf.image.resize(x, (IMG,IMG)).numpy().astype('uint8') for x,_ in test.take(4)]
gr.Interface(predict, gr.Image(), gr.Label(), title='MalariaLens',
             description='Upload a single blood-cell image to check for malaria parasites.',
             examples=examples).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2dfee9932e5baff870.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
